# SLM from-scratch training on Colab (A100): 100B tokens, everything on Google Drive

**Model:** 20 layers, d_model 640, FFN 2048, 10 heads / 2 KV heads, vocab 32,768, max context 16,384 (~141M parameters).
**Data:** `EleutherAI/SmolLM2-135M-100B`, streamed and tokenized on the fly into 2B-token shards on Drive (~200 GB). Raw text is never stored.

**Workflow**
1. **Part 0**: setup + config + source files. Re-run it on every reconnect.
2. **Part 1**: train the tokenizer, then tokenize to shards. CPU-bound, resumable, takes a long time.
3. **Part 2**: training on the A100. It can **start as soon as the first shard exists**; if training catches up with tokenization it waits for the next shard. Resumes automatically from the latest checkpoint.

You can run Part 1 and Part 2 at the same time in two Colab sessions (same Drive). Tokenize on a CPU runtime and train on the A100 one.

> After the first training step, do not change anything in the "training run" block of the config cell. The training script refuses to resume if settings differ.

## Part 0: Setup

In [ ]:
#@title 0.1 Install + mount Google Drive
!pip install -q -U datasets tokenizers

from google.colab import drive
drive.mount("/content/drive")   # sign in with the account whose Drive should hold the data

In [ ]:
#@title 0.2 Config (edit here only)
import os, time, json, pickle, shutil, subprocess, math
from collections import Counter

# ---------- Drive layout ----------
PROJECT_DIR = "/content/drive/MyDrive/slm_4k"
SHARD_DIR   = f"{PROJECT_DIR}/data/tokenized/train_shards"
for sub in ["tokenizer", "data/tokenized/train_shards", "checkpoints", "logs"]:
    os.makedirs(f"{PROJECT_DIR}/{sub}", exist_ok=True)
os.chdir(PROJECT_DIR)          # train_4k.py writes checkpoints/ and logs/ relative to cwd

TOK_PATH   = f"{PROJECT_DIR}/tokenizer/tokenizer.json"
VAL_BIN    = f"{PROJECT_DIR}/data/tokenized/val.bin"
STATE_PATH = f"{PROJECT_DIR}/data/tokenized/prep_state.pkl"
PLAN_PATH  = f"{PROJECT_DIR}/run_plan.json"

# ---------- Dataset ----------
DATASET = "EleutherAI/SmolLM2-135M-100B"
SOURCES = None     # None = all sources, or e.g. ["fineweb_edu", "dclm_edu", "cosmopedia_v2", "stack_edu"]

# ---------- Tokenizer ----------
VOCAB_SIZE = 32768
TOK_TRAIN_DOCS = 500_000        # streamed docs used to train the BPE tokenizer
TOK_MAX_CHARS = 20_000          # per-doc character cap while training the tokenizer
FORCE_RETRAIN_TOKENIZER = False

# ---------- Data budget ----------
TRAIN_TOKENS_TARGET = 100_000_000_000   # 100B train tokens (~200 GB on Drive)
SHARD_TOKENS = 2_000_000_000            # 2B tokens = 4 GB per shard file
VAL_MAX_TOKENS = 20_000_000
BATCH_DOCS = 1000
SAVE_STATE_EVERY_BATCHES = 50           # ~50M tokens between saves of the resume point

# ---------- Training run (do not change after the first step) ----------
TAG = "scratch4k"
SEQ = 4096          # training sequence length. Model supports 16384; see note in Part 2
MICRO_BS = 4
ACCUM = 32          # 4 x 32 x 4096 = 524,288 tokens per step
LR = 6e-4
WARMUP_STEPS = 2000
EVAL_EVERY = 500
SAVE_EVERY = 500    # keep equal to EVAL_EVERY so the "latest" checkpoint is never older than "best"
SEED = 1
REASONING_PCT = 0   # no reasoning dataset in this run

tokens_per_step = MICRO_BS * ACCUM * SEQ
STEPS = (TRAIN_TOKENS_TARGET - 1) // tokens_per_step    # every block is used exactly once

# ---------- Local cache for shards during training ----------
SHARD_CACHE_DIR = "/content/shard_cache"
LOCAL_DIR = "/content/local_data"

# ---------- Optional HF token (strongly recommended for a multi-day stream) ----------
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

print(f"CPU cores         : {os.cpu_count()}")
print(f"Tokens per step   : {tokens_per_step:,}")
print(f"Total steps       : {STEPS:,}")
print(f"Train data on Drive: ~{TRAIN_TOKENS_TARGET*2/1e9:.0f} GB  ({math.ceil(TRAIN_TOKENS_TARGET/SHARD_TOKENS)} shards)")
try:
    print(f"Drive free space  : {shutil.disk_usage(PROJECT_DIR).free/1e9:.0f} GB")
except Exception:
    pass

### Source files
These cells write the model, training script and shard reader into the Drive project folder.
`model_4k.py` is yours with `ffn_dim = 2048` (the only change). `train_4k.py` is yours with two small patches (marked with comments): it reads a directory of shards, and `SEQUENTIAL_BLOCKS=1` makes it consume blocks in order instead of shuffling them.

In [ ]:
%%writefile model_4k.py
'''

@dataclass
class Config:
    vocab_size: int = 16384 # number of tokens tokenizer know
    d_model: int = 512 # size of token representation
    n_layers: int = 20 # number of transformer blocks
    n_heads: int = 8 # Query attention heads
    n_kv_heads: int = 2 # Key/Value attention heads
    head_dim: int = 64 # Attention head dimension d_model = n_heads * head_dim => 8 * 64 = 512
    ffn_dim: int = 1408 # Fead Forward Network dimension
    ple_dim: int = 32 # Size of PLE representation
    max_seq: int = 2048 # max supported sequence length
    rope_theta: float = 10000.0 # RoPE frequency
    norm_eps: float = 1e-6 # numerical stability constant (prevent division by zero)

'''
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 32768
    d_model: int = 640
    n_layers: int = 20
    n_heads: int = 10
    n_kv_heads: int = 2
    head_dim: int = 64
    ffn_dim: int = 2048
    ple_dim: int = 32
    max_seq: int = 16384
    rope_theta: float = 10000.0
    norm_eps: float = 1e-6

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        mean_square = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(mean_square + self.eps)
        return x * self.weight

def build_rope(max_seq, head_dim, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(max_seq).float()
    angles = torch.outer(positions, inv_freq)
    angles = torch.cat([angles, angles], dim=1)
    return angles.cos(), angles.sin()

def rotate_half(x):
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * cfg.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.q_norm = RMSNorm(cfg.head_dim, cfg.norm_eps)
        self.k_norm = RMSNorm(cfg.head_dim, cfg.norm_eps)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        c = self.cfg

        q = self.q_proj(x).view(B, T, c.n_heads, c.head_dim).transpose(1,2)
        k = self.k_proj(x).view(B, T, c.n_kv_heads, c.head_dim).transpose(1,2)
        v = self.v_proj(x).view(B, T, c.n_kv_heads, c.head_dim).transpose(1,2)

        q = self.q_norm(q)
        k = self.k_norm(k)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        group = c.n_heads // c.n_kv_heads
        k = k.repeat_interleave(group, dim=1)
        v = v.repeat_interleave(group, dim=1)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1,2).reshape(B,T, c.n_heads * c.head_dim)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.up_proj = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.down_proj = nn.Linear(cfg.ffn_dim, cfg.d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg)
        self.ple_gate = nn.Linear(cfg.d_model, cfg.ple_dim, bias=False)
        self.ple_up = nn.Linear(cfg.ple_dim, cfg.d_model, bias=False)
        self.ple_norm = RMSNorm(cfg.d_model, cfg.norm_eps)

    def forward(self, x, cos, sin, ple):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        g = F.gelu(self.ple_gate(x), approximate="tanh") * ple
        x = x + self.ple_norm(self.ple_up(g))
        return x

class SLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.ple_table = nn.Embedding(cfg.vocab_size, cfg.n_layers * cfg.ple_dim)
        self.ple_proj = nn.Linear(cfg.d_model, cfg.n_layers * cfg.ple_dim, bias=False)
        self.ple_proj_norm = RMSNorm(cfg.ple_dim, cfg.norm_eps)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head= nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

        cos, sin = build_rope(cfg.max_seq, cfg.head_dim, cfg.rope_theta)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith(("o_proj.weight", "down_proj.weight", "ple_up.weight")):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, T = ids.shape
        cfg = self.cfg

        x = self.tok_emb(ids)

        ple_tok = self.ple_table(ids).view(B, T, cfg.n_layers, cfg.ple_dim)
        ple_tok = ple_tok * math.sqrt(cfg.ple_dim)
        ple_ctx = self.ple_proj(x).view(B, T, cfg.n_layers, cfg.ple_dim)
        ple_ctx = self.ple_proj_norm(ple_ctx)
        ple = (ple_tok + ple_ctx) / math.sqrt(2.0)

        cos, sin = self.cos[:T], self.sin[:T]
        for i, block in enumerate(self.blocks):
            x = block(x, cos, sin, ple[:, :, i])

        x = self.final_norm(x)
        return self.lm_head(x)


if __name__ == "__main__":
    cfg = Config()
    model = SLM(cfg)

    total = sum(p.numel() for p in model.parameters())
    ple_params = model.ple_table.weight.numel()
    print(f"total parameters : {total:,}")
    print(f"PLE lookup table : {ple_params:,}")
    print(f"without PLE table: {total - ple_params:,}")
    print("weights tied     :", model.lm_head.weight is model.tok_emb.weight)

    ids = torch.randint(0, cfg.vocab_size, (2, 32))
    logits = model(ids)
    print("logits shape     :", tuple(logits.shape))

    targets = torch.randint(0, cfg.vocab_size, (2, 32))
    loss = F.cross_entropy(logits.view(-1, cfg.vocab_size), targets.view(-1))
    print(f"initial loss     : {loss.item():.3f}  (random guessing = {math.log(cfg.vocab_size):.3f})")

In [ ]:
%%writefile sharded_data.py
"""Read-only, ndarray-like view over many uint16 token shards stored on Google Drive.

Designed for SEQUENTIAL consumption (train_4k.py with SEQUENTIAL_BLOCKS=1):
  * only the previous, current and next shard live on the local SSD
  * the next shard is copied from Drive in a background thread
  * if the tokenizer is still producing shards, reads wait until the shard is complete
Supports len() and slicing (including slices that straddle two shards).
"""
import json
import os
import shutil
import threading
import time

import numpy as np


class ShardedTokens:

    def __init__(self, shard_dir, cache_dir=None, poll_s=60):
        self.dir = shard_dir
        self.cache = cache_dir or os.environ.get("SHARD_CACHE_DIR", "/content/shard_cache")
        os.makedirs(self.cache, exist_ok=True)
        m = self._manifest()
        self.S = int(m["shard_tokens"])
        self.pattern = m.get("pattern", "shard_{:05d}.bin")
        self.total = int(m["planned_tokens"])
        self.poll_s = poll_s
        self._mm = {}
        self._cur = -1
        self._lock = threading.Lock()
        self._pf = None

    # ------------------------------------------------------------------
    def _manifest(self):
        with open(os.path.join(self.dir, "manifest.json")) as f:
            return json.load(f)

    def _name(self, k):
        return self.pattern.format(k)

    def __len__(self):
        return self.total

    def _expected_tokens(self, k):
        m = self._manifest()
        if m.get("done"):
            actual = int(m["actual_tokens"])
            if k * self.S >= actual:
                raise IndexError(f"shard {k} is beyond the {actual:,} tokens that were prepared")
            return min(self.S, actual - k * self.S)
        return self.S   # while preparation runs, a shard is ready once it is full

    # ------------------------------------------------------------------
    def _ensure_local(self, k):
        name = self._name(k)
        src = os.path.join(self.dir, name)
        dst = os.path.join(self.cache, name)
        waited = 0
        # wait (without holding the lock) until the shard is complete on Drive
        while True:
            need = self._expected_tokens(k) * 2
            if os.path.exists(dst) and os.path.getsize(dst) == need:
                return dst
            if os.path.exists(src) and os.path.getsize(src) == need:
                break
            if waited % 5 == 0:
                print(f"[shards] waiting for {name} to be complete on Drive ...", flush=True)
            waited += 1
            time.sleep(self.poll_s)
        # the lock only serialises the copy itself
        with self._lock:
            if os.path.exists(dst) and os.path.getsize(dst) == need:
                return dst   # another thread already cached it
            tmp = dst + ".part"
            t0 = time.time()
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
            print(f"[shards] cached {name} ({need/1e9:.1f} GB) in {time.time()-t0:.0f}s", flush=True)
            return dst

    def _prefetch(self, k):
        try:
            self._ensure_local(k)
        except Exception:
            pass   # past the end, or will be retried on demand

    def _advance(self, k):
        self._cur = k
        for j in [j for j in self._mm if j < k - 1]:   # keep k-1: the y-slice of a boundary block still reads it
            del self._mm[j]
        for fn in os.listdir(self.cache):
            if fn.endswith(".bin"):
                try:
                    idx = int(fn.split("_")[1].split(".")[0])
                except (IndexError, ValueError):
                    continue
                if idx < k - 1:
                    try:
                        os.remove(os.path.join(self.cache, fn))
                    except OSError:
                        pass
        if self._pf is None or not self._pf.is_alive():
            self._pf = threading.Thread(target=self._prefetch, args=(k + 1,), daemon=True)
            self._pf.start()

    def _shard(self, k):
        mm = self._mm.get(k)
        if mm is None:
            path = self._ensure_local(k)
            mm = np.memmap(path, dtype=np.uint16, mode="r")
            self._mm[k] = mm
        if k > self._cur:
            self._advance(k)
        return mm

    # ------------------------------------------------------------------
    def __getitem__(self, key):
        if not isinstance(key, slice):
            raise TypeError("ShardedTokens only supports slicing")
        start, stop, _ = key.indices(self.total)
        parts = []
        pos = start
        while pos < stop:
            k = pos // self.S
            off = pos - k * self.S
            n = min(stop - pos, self.S - off)
            parts.append(self._shard(k)[off:off + n])
            pos += n
        if not parts:
            return np.zeros(0, dtype=np.uint16)
        return parts[0] if len(parts) == 1 else np.concatenate(parts)


def open_train_data(path):
    """Directory of shards -> ShardedTokens; single file -> plain memmap."""
    if os.path.isdir(path):
        return ShardedTokens(path)
    return np.memmap(path, dtype=np.uint16, mode="r")

In [ ]:
%%writefile train_4k.py
import argparse
import math
import os
import json
import time

from datetime import datetime
from dataclasses import asdict

import numpy as np
import torch
import torch.nn.functional as F

from model_4k import Config, SLM


# ============================================================
# 1. ARGUMENTS
# ============================================================

parser = argparse.ArgumentParser(
    description="Train from scratch on FineWeb 10BT mixed with reasoning data."
)

parser.add_argument("--steps", type=int, default=166000)
parser.add_argument("--micro_bs", type=int, default=1)
parser.add_argument("--accum", type=int, default=16)
parser.add_argument("--seq", type=int, default=4096)

parser.add_argument("--lr", type=float, default=6e-4)
parser.add_argument("--warmup_steps", type=int, default=500)

parser.add_argument("--eval_every", type=int, default=500)
parser.add_argument("--save_every", type=int, default=500)

parser.add_argument(
    "--unseen",
    type=str,
    default="data/tokenized/all_blocks_4096.npy",
    help="FineWeb block indices to read once each (scripts/make_block.py), or 'none' for random sampling."
)

parser.add_argument("--seed", type=int, default=1)

parser.add_argument(
    "--fill_seen",
    action="store_true",
    help="Use previously seen FineWeb blocks if unseen blocks run out."
)

parser.add_argument(
    "--tag",
    type=str,
    default="scratch4k",
    help="Prefix for this run's checkpoints and logs."
)

parser.add_argument(
    "--train_bin",
    type=str,
    default="data/tokenized/train.bin"
)

parser.add_argument(
    "--val_bin",
    type=str,
    default="data/tokenized/val.bin"
)

# New names are reasoning_pct and reasoning_bin.
# Old math_pct and math_bin names remain accepted for compatibility.

parser.add_argument(
    "--reasoning_pct",
    "--math_pct",
    dest="reasoning_pct",
    type=int,
    default=3,
    help="Percentage of training sequences sampled from reasoning data."
)

parser.add_argument(
    "--reasoning_bin",
    "--math_bin",
    dest="reasoning_bin",
    type=str,
    default="data/tokenized/reasoning_synth.bin",
    help="Combined tokenized reasoning dataset."
)

parser.add_argument(
    "--reasoning_val_tokens",
    type=int,
    default=500_000,
    help="Approximate number of final reasoning tokens reserved for validation."
)

parser.add_argument(
    "--reasoning_eos_id",
    type=int,
    default=2,
    help="EOS token ID used to split reasoning examples."
)

parser.add_argument(
    "--resume",
    type=str,
    default=None,
    help="Resume this run from its own checkpoint."
)

args = parser.parse_args()


# ============================================================
# 2. ARGUMENT VALIDATION
# ============================================================

if args.steps <= 0:
    raise ValueError("--steps must be positive.")

if args.micro_bs <= 0 or args.accum <= 0 or args.seq <= 0:
    raise ValueError("Batch sizes and sequence length must be positive.")

if args.lr <= 0:
    raise ValueError("--lr must be positive.")

if args.warmup_steps < 0:
    raise ValueError("--warmup_steps cannot be negative.")

if args.eval_every <= 0 or args.save_every <= 0:
    raise ValueError("Evaluation and save intervals must be positive.")

if not 0 <= args.reasoning_pct < 100:
    raise ValueError("--reasoning_pct must be between 0 and 99.")

if args.reasoning_val_tokens <= args.seq + 1:
    raise ValueError("--reasoning_val_tokens must exceed seq + 1.")

if args.reasoning_eos_id < 0 or args.reasoning_eos_id >= 65536:
    raise ValueError("Invalid EOS token ID for uint16 data.")


# ============================================================
# 3. DEVICE AND RANDOM SEEDS
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

device = "cuda"

torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)

rng = np.random.default_rng(args.seed)
val_rng = np.random.default_rng(1)

# Prefer BF16 where supported.
# Otherwise use full precision rather than risk FP16 overflow
# without a gradient scaler.

use_bf16 = torch.cuda.is_bf16_supported()

if use_bf16:
    amp_enabled = True
    amp_dtype = torch.bfloat16
    amp_name = "bfloat16"
else:
    amp_enabled = False
    amp_dtype = torch.float32
    amp_name = "float32"


# ============================================================
# 4. OUTPUT PATHS AND LOGGING
# ============================================================

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

latest_path = f"checkpoints/{args.tag}_latest.pt"

best_fineweb_path = f"checkpoints/{args.tag}_best_fineweb.pt"
best_reasoning_path = f"checkpoints/{args.tag}_best_reasoning.pt"

log_json = f"logs/{args.tag}.json"
log_file = f"logs/{args.tag}.log"

if args.resume is None and os.path.exists(latest_path):
    raise RuntimeError(
        f"{latest_path} already exists.\n"
        f"Use --resume {latest_path} to resume it, "
        f"or choose another --tag."
    )

tokens_per_step = args.micro_bs * args.accum * args.seq
total_tokens = tokens_per_step * args.steps


# ============================================================
# 5. CONFIGURATION RECORD
# ============================================================

run_config = {
    "steps": args.steps,
    "micro_bs": args.micro_bs,
    "accum": args.accum,
    "seq": args.seq,
    "lr": args.lr,
    "warmup_steps": args.warmup_steps,
    "eval_every": args.eval_every,
    "save_every": args.save_every,
    "unseen": args.unseen,
    "seed": args.seed,
    "fill_seen": args.fill_seen,
    "tag": args.tag,
    "reasoning_pct": args.reasoning_pct,
    "reasoning_bin": args.reasoning_bin,
    "reasoning_val_tokens": args.reasoning_val_tokens,
    "reasoning_eos_id": args.reasoning_eos_id,
    "train_bin": args.train_bin,
    "val_bin": args.val_bin,
}


training_history = {
    "start_time": datetime.now().isoformat(),
    "config": run_config,
    "metrics": []
}


def save_metrics():

    temp_path = log_json + ".tmp"

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(training_history, f, indent=2)

    os.replace(temp_path, log_json)


def write_log(message):

    print(message, flush=True)

    with open(log_file, "a", encoding="utf-8") as f:
        f.write(message + "\n")


# ============================================================
# 6. MODEL
# ============================================================

cfg = Config()

if args.seq > cfg.max_seq:
    raise ValueError(
        f"--seq {args.seq} is larger than Config.max_seq {cfg.max_seq}. "
        "Raise max_seq in model.py."
    )

model = SLM(cfg).to(device)

n_params = sum(p.numel() for p in model.parameters())


# ============================================================
# 7. DATASETS
# ============================================================

if not os.path.exists(args.train_bin):
    raise FileNotFoundError(args.train_bin)

if not os.path.exists(args.val_bin):
    raise FileNotFoundError(args.val_bin)

# train_bin may be a directory of shards (see sharded_data.py) or a single .bin file
from sharded_data import open_train_data

train_data = open_train_data(args.train_bin)

val_data = np.memmap(
    args.val_bin,
    dtype=np.uint16,
    mode="r"
)

if len(train_data) <= args.seq + 1:
    raise ValueError("FineWeb training dataset is too small.")

if len(val_data) <= args.seq + 1:
    raise ValueError("FineWeb validation dataset is too small.")


# ------------------------------------------------------------
# 7A. REASONING DATA
# ------------------------------------------------------------

reasoning_data = None

n_reasoning_train = 0
reasoning_val_start = 0
reasoning_val_length = 0

if args.reasoning_pct > 0:

    if not os.path.exists(args.reasoning_bin):
        raise FileNotFoundError(args.reasoning_bin)

    reasoning_data = np.memmap(
        args.reasoning_bin,
        dtype=np.uint16,
        mode="r"
    )

    if len(reasoning_data) <= args.seq + args.reasoning_val_tokens:
        raise ValueError(
            "Reasoning dataset is too small for the requested "
            "training and validation split."
        )

    # Approximate target position for validation.
    target_start = max(
        0,
        len(reasoning_data) - args.reasoning_val_tokens
    )

    # Find the first EOS at or after target_start.
    # Validation begins immediately after that EOS.
    #
    # This prevents a training example from being split
    # between the training and validation regions.

    eos_positions = np.flatnonzero(
        reasoning_data[target_start:] == args.reasoning_eos_id
    )

    if len(eos_positions) == 0:
        raise RuntimeError(
            "Could not find an EOS token after the reasoning "
            "validation split position. Check reasoning_eos_id "
            "and your tokenized dataset."
        )

    reasoning_val_start = (
        target_start + int(eos_positions[0]) + 1
    )

    n_reasoning_train = reasoning_val_start

    reasoning_val_length = (
        len(reasoning_data) - reasoning_val_start
    )

    if n_reasoning_train <= args.seq + 1:
        raise ValueError("Reasoning training portion is too small.")

    if reasoning_val_length <= args.seq + 1:
        raise ValueError("Reasoning validation portion is too small.")

    write_log(
        f"Reasoning dataset tokens: {len(reasoning_data):,}"
    )

    write_log(
        f"Reasoning training tokens: {n_reasoning_train:,}"
    )

    write_log(
        f"Reasoning validation tokens: {reasoning_val_length:,}"
    )

    write_log(
        f"Reasoning validation starts at token: {reasoning_val_start:,}"
    )

    write_log(
        f"Reasoning EOS ID: {args.reasoning_eos_id}"
    )


# ============================================================
# 8. DATASET SAMPLING
# ============================================================

def is_reasoning_seq(i):

    # Exactly reasoning_pct percent over every 100 sequences.
    #
    # Example for 10%:
    # One reasoning sequence approximately every 10 sequences.
    #
    # The distribution is deterministic and resumes from seq_idx.

    return (
        ((i + 1) * args.reasoning_pct) // 100
        >
        (i * args.reasoning_pct) // 100
    )


# ------------------------------------------------------------
# 8A. UNSEEN FINEWEB BLOCKS
# ------------------------------------------------------------

order = None
unseen_blocks = None

sample_pos = 0
seq_idx = 0

needed = 0

if args.unseen.lower() != "none":

    if not os.path.exists(args.unseen):
        raise FileNotFoundError(
            f"{args.unseen} not found. "
            "Run scripts/make_block.py first."
        )

    unseen_blocks = np.load(args.unseen)

    if unseen_blocks.ndim != 1:
        raise ValueError("Unseen block file must be one-dimensional.")

    if len(unseen_blocks) == 0:
        raise ValueError("No unseen FineWeb blocks were found.")

    if not np.issubdtype(unseen_blocks.dtype, np.integer):
        raise ValueError("Unseen block indices must be integers.")

    # Number of blocks that can hold a full input sequence
    # plus its final target token.

    n_usable = (len(train_data) - 1) // args.seq

    if np.any(unseen_blocks < 0) or np.any(unseen_blocks >= n_usable):
        raise ValueError(
            "Unseen block file contains invalid block indices "
            "for the current FineWeb dataset or sequence length."
        )

    if len(np.unique(unseen_blocks)) != len(unseen_blocks):
        raise ValueError("Unseen block file contains duplicate indices.")

    order_rng = np.random.default_rng(args.seed)

    # Shuffle fresh blocks once.
    # SEQUENTIAL_BLOCKS=1: consume blocks in file order (the data stream is already shuffled).
    # Needed so only a few shards have to be on the local disk at a time.
    order = (unseen_blocks if os.environ.get("SEQUENTIAL_BLOCKS") == "1"
             else order_rng.permutation(unseen_blocks))

    if args.fill_seen:

        seen_blocks = np.setdiff1d(
            np.arange(n_usable, dtype=np.int64),
            unseen_blocks
        )

        order = np.concatenate([
            order,
            order_rng.permutation(seen_blocks)
        ])

    total_seqs = args.steps * args.accum * args.micro_bs

    needed = (
        total_seqs
        -
        (total_seqs * args.reasoning_pct) // 100
    )

    if needed > len(order):
        raise ValueError(
            f"Not enough FineWeb blocks.\n"
            f"Required: {needed:,}\n"
            f"Available: {len(order):,}\n"
            "Reduce steps or enable --fill_seen."
        )

else:

    write_log(
        "WARNING: FineWeb sampling uses random windows. "
        "Some pretraining data may repeat."
    )


# ============================================================
# 9. BATCH GENERATION
# ============================================================

def make_batch(data, indices):

    x = np.stack([
        data[i:i + args.seq] for i in indices
    ]).astype(np.int64)

    y = np.stack([
        data[i + 1:i + 1 + args.seq] for i in indices
    ]).astype(np.int64)

    x = torch.from_numpy(x).to(device)
    y = torch.from_numpy(y).to(device)

    return x, y


def get_train_batch():

    global sample_pos, seq_idx

    xs = []
    ys = []

    for _ in range(args.micro_bs):

        if is_reasoning_seq(seq_idx):

            # Sample a random reasoning window.
            # The upper bound keeps the window inside the
            # training portion of the reasoning dataset.

            start = int(
                rng.integers(
                    0,
                    n_reasoning_train - args.seq - 1
                )
            )

            source = reasoning_data

        elif order is not None:

            # Each selected FineWeb block is used once.
            start = int(order[sample_pos]) * args.seq

            sample_pos += 1

            source = train_data

        else:

            # Random FineWeb sampling if unseen blocks are disabled.
            start = int(
                rng.integers(
                    0,
                    len(train_data) - args.seq - 1
                )
            )

            source = train_data

        xs.append(source[start:start + args.seq])

        ys.append(
            source[start + 1:start + 1 + args.seq]
        )

        seq_idx += 1

    x = torch.from_numpy(
        np.stack(xs).astype(np.int64)
    ).to(device)

    y = torch.from_numpy(
        np.stack(ys).astype(np.int64)
    ).to(device)

    return x, y


def get_val_batch(batch_rng):

    indices = batch_rng.integers(
        0,
        len(val_data) - args.seq - 1,
        size=args.micro_bs
    )

    return make_batch(val_data, indices)


def get_reasoning_val_batch(batch_rng):

    indices = (
        reasoning_val_start
        +
        batch_rng.integers(
            0,
            reasoning_val_length - args.seq - 1,
            size=args.micro_bs
        )
    )

    return make_batch(reasoning_data, indices)


# ============================================================
# 10. LOSS
# ============================================================

def compute_loss(x, y):

    with torch.autocast(
        device_type="cuda",
        dtype=amp_dtype,
        enabled=amp_enabled
    ):

        logits = model(x)

    loss = F.cross_entropy(
        logits.float().reshape(-1, cfg.vocab_size),
        y.reshape(-1)
    )

    return loss


# ============================================================
# 11. OPTIMIZER
# ============================================================

decay = []
no_decay = []

for name, p in model.named_parameters():

    if p.dim() >= 2 and "emb" not in name and "table" not in name:
        decay.append(p)
    else:
        no_decay.append(p)


opt = torch.optim.AdamW(
    [
        {
            "params": decay,
            "weight_decay": 0.1
        },
        {
            "params": no_decay,
            "weight_decay": 0.0
        }
    ],
    lr=args.lr,
    betas=(0.9, 0.95),
    fused=True
)


# ============================================================
# 12. LEARNING RATE SCHEDULER
# ============================================================

warmup = max(
    1,
    min(args.warmup_steps, args.steps)
)

min_lr_ratio = 0.1


def get_lr(step):

    if step < warmup:
        return args.lr * (step + 1) / warmup

    progress = (
        (step - warmup)
        /
        max(1, args.steps - warmup)
    )

    progress = min(1.0, max(0.0, progress))

    return args.lr * (
        min_lr_ratio
        +
        (1 - min_lr_ratio)
        * 0.5
        * (1 + math.cos(math.pi * progress))
    )


# ============================================================
# 13. VALIDATION
# ============================================================

@torch.no_grad()
def evaluate_fineweb(n_batches=50):

    model.eval()

    total = 0.0

    fixed_rng = np.random.default_rng(1)

    for _ in range(n_batches):

        x, y = get_val_batch(fixed_rng)

        loss = compute_loss(x, y)

        total += loss.item()

    model.train()

    return total / n_batches


@torch.no_grad()
def evaluate_reasoning(n_batches=20):

    if reasoning_data is None:
        return None

    model.eval()

    total = 0.0

    fixed_rng = np.random.default_rng(2)

    for _ in range(n_batches):

        x, y = get_reasoning_val_batch(fixed_rng)

        loss = compute_loss(x, y)

        total += loss.item()

    model.train()

    return total / n_batches


# ============================================================
# 14. CHECKPOINT STATE
# ============================================================

start_step = 0

best_fineweb_loss = float("inf")
best_reasoning_loss = float("inf")

last_fineweb_loss = None
last_reasoning_loss = None


# ============================================================
# 15. CHECKPOINT SAVING
# ============================================================

def save_checkpoint(path, step):

    torch.cuda.synchronize()

    checkpoint = {

        "model": model.state_dict(),

        "optimizer": opt.state_dict(),

        "cfg": asdict(cfg),

        "step": step,

        "val_loss": last_fineweb_loss,
        "best_val_loss": best_fineweb_loss,

        "fineweb_val_loss": last_fineweb_loss,
        "best_fineweb_loss": best_fineweb_loss,

        "reasoning_val_loss": last_reasoning_loss,
        "best_reasoning_loss": best_reasoning_loss,

        "train_rng_state": rng.bit_generator.state,
        "val_rng_state": val_rng.bit_generator.state,

        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),

        "training_history": training_history,

        "run_config": run_config,

        "sample_pos": sample_pos,
        "seq_idx": seq_idx,
    }

    temp_path = path + ".tmp"

    torch.save(checkpoint, temp_path)

    os.replace(temp_path, path)

    write_log(f"Checkpoint saved: {path}")


# ============================================================
# 16. INITIALIZE OR RESUME
# ============================================================

if args.resume is not None:

    if not os.path.exists(args.resume):
        raise FileNotFoundError(args.resume)

    write_log(f"Loading checkpoint: {args.resume}")

    checkpoint = torch.load(
        args.resume,
        map_location=device,
        weights_only=False
    )

    required_keys = [
        "model",
        "optimizer",
        "step",
        "train_rng_state",
        "torch_rng_state",
        "cuda_rng_state",
        "run_config",
        "sample_pos",
        "seq_idx",
    ]

    missing_keys = [
        key for key in required_keys
        if key not in checkpoint
    ]

    if missing_keys:
        raise RuntimeError(
            "Checkpoint is missing required resume information: "
            f"{missing_keys}"
        )

    if checkpoint["cfg"] != asdict(cfg):
        raise RuntimeError(
            "Model configuration differs from the checkpoint."
        )

    # Require identical settings on resume.
    # This prevents changing the data mixture or scheduler mid-run.

    saved_config = checkpoint["run_config"]

    if saved_config != run_config:
        differences = []

        for key in run_config:
            if saved_config.get(key) != run_config[key]:
                differences.append(
                    f"{key}: saved={saved_config.get(key)!r}, "
                    f"current={run_config[key]!r}"
                )

        raise RuntimeError(
            "Training configuration differs from the checkpoint:\n"
            + "\n".join(differences)
            + "\nUse the original settings when resuming."
        )

    model.load_state_dict(checkpoint["model"])

    opt.load_state_dict(checkpoint["optimizer"])

    start_step = int(checkpoint["step"])

    best_fineweb_loss = checkpoint.get(
        "best_fineweb_loss",
        checkpoint.get("best_val_loss", float("inf"))
    )

    best_reasoning_loss = checkpoint.get(
        "best_reasoning_loss",
        float("inf")
    )

    last_fineweb_loss = checkpoint.get(
        "fineweb_val_loss",
        checkpoint.get("val_loss")
    )

    last_reasoning_loss = checkpoint.get(
        "reasoning_val_loss"
    )

    rng.bit_generator.state = checkpoint["train_rng_state"]

    if "val_rng_state" in checkpoint:
        val_rng.bit_generator.state = checkpoint["val_rng_state"]

    torch.set_rng_state(
        checkpoint["torch_rng_state"].cpu()
    )

    torch.cuda.set_rng_state_all(
        [
            state.cpu()
            for state in checkpoint["cuda_rng_state"]
        ]
    )

    training_history = checkpoint.get(
        "training_history",
        training_history
    )

    sample_pos = int(checkpoint["sample_pos"])
    seq_idx = int(checkpoint["seq_idx"])

    if start_step >= args.steps:
        raise ValueError(
            "Checkpoint has already reached the requested "
            "training step count."
        )

    expected_seq_idx = (
        start_step * args.accum * args.micro_bs
    )

    if seq_idx != expected_seq_idx:
        raise RuntimeError(
            "Checkpoint sequence counter does not match its step."
        )

    expected_sample_pos = (
        seq_idx
        -
        (seq_idx * args.reasoning_pct) // 100
    )

    if sample_pos != expected_sample_pos and order is not None:
        raise RuntimeError(
            "Checkpoint FineWeb sampling position is inconsistent."
        )

    write_log(f"Resuming from step {start_step:,}")


else:

    write_log(
        f"Training from scratch (random initialization), "
        f"{n_params:,} parameters, d_model={cfg.d_model}, "
        f"max_seq={cfg.max_seq}"
    )


# ============================================================
# 17. TRAINING INFORMATION
# ============================================================

print("\n========== FROM-SCRATCH TRAINING CONFIGURATION ==========")

print(f"Tag: {args.tag}")

print(
    "Initialization: "
    + ("random weights" if args.resume is None else f"resumed from {args.resume}")
)

print(f"Model parameters: {n_params:,}")

print(f"Total training steps: {args.steps:,}")
print(f"Starting step: {start_step:,}")

print(f"Micro batch: {args.micro_bs}")
print(f"Gradient accumulation: {args.accum}")
print(f"Sequence length: {args.seq}")

print(f"Tokens per step: {tokens_per_step:,}")
print(f"Planned total tokens: {total_tokens:,}")

print(f"FineWeb training data: {args.train_bin}")
print(f"FineWeb validation data: {args.val_bin}")

if order is not None:

    print(f"Unseen blocks: {len(unseen_blocks):,}")
    print(f"Blocks available: {len(order):,}")
    print(f"FineWeb blocks required: {needed:,}")

    print(
        f"FineWeb blocks remaining after planned run: "
        f"{len(order) - needed:,}"
    )

else:

    print("FineWeb sampling: random windows")

print(f"Reasoning percentage: {args.reasoning_pct}%")

if reasoning_data is not None:

    print(f"Reasoning data: {args.reasoning_bin}")
    print(f"Reasoning training tokens: {n_reasoning_train:,}")
    print(f"Reasoning validation tokens: {reasoning_val_length:,}")

print(f"Peak learning rate: {args.lr}")
print(f"Warmup steps: {warmup}")

print(f"AMP precision: {amp_name}")

print(f"Validation interval: {args.eval_every}")
print(f"Checkpoint interval: {args.save_every}")

print("======================================================\n")


# ============================================================
# 18. INITIAL BASELINE VALIDATION
# ============================================================

if start_step == 0:

    initial_fineweb = evaluate_fineweb()

    write_log(
        f">>> step 0 | FineWeb validation loss "
        f"{initial_fineweb:.4f} (starting weights)"
    )

    training_history["initial_fineweb_val_loss"] = initial_fineweb

    if reasoning_data is not None:

        initial_reasoning = evaluate_reasoning()

        write_log(
            f">>> step 0 | Reasoning validation loss "
            f"{initial_reasoning:.4f} (starting weights)"
        )

        training_history["initial_reasoning_val_loss"] = (
            initial_reasoning
        )

    save_metrics()


# ============================================================
# 19. TRAINING LOOP
# ============================================================

model.train()

torch.cuda.reset_peak_memory_stats()

t0 = time.time()

for step in range(start_step, args.steps):

    # --------------------------------------------------------
    # Learning rate
    # --------------------------------------------------------

    lr = get_lr(step)

    for group in opt.param_groups:
        group["lr"] = lr

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------

    opt.zero_grad(set_to_none=True)

    step_loss = 0.0

    for _ in range(args.accum):

        x, y = get_train_batch()

        loss = compute_loss(x, y) / args.accum

        loss.backward()

        step_loss += loss.detach()

    # --------------------------------------------------------
    # Gradient clipping
    # --------------------------------------------------------

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    # --------------------------------------------------------
    # Optimizer update
    # --------------------------------------------------------

    opt.step()

    global_step = step + 1

    # --------------------------------------------------------
    # Progress logging
    # --------------------------------------------------------

    should_log = (
        global_step % 10 == 0
        or global_step == args.steps
    )

    if should_log:

        elapsed = time.time() - t0

        completed_steps = global_step - start_step

        speed = (
            tokens_per_step * completed_steps
            /
            max(elapsed, 1e-8)
        )

        mem = (
            torch.cuda.max_memory_allocated()
            /
            1e9
        )

        eta_h = (
            elapsed
            /
            max(completed_steps, 1)
            *
            (args.steps - global_step)
            /
            3600
        )

        train_loss = step_loss.item()

        grad_value = grad_norm.item()

        message = (
            f"step {global_step:6d} | "
            f"loss {train_loss:.4f} | "
            f"lr {lr:.2e} | "
            f"grad {grad_value:.2f} | "
            f"{speed:,.0f} tok/s | "
            f"{mem:.2f} GB | "
            f"eta {eta_h:.1f} h"
        )

        write_log(message)

        training_history["metrics"].append({

            "step": global_step,

            "train_loss": train_loss,

            "learning_rate": lr,

            "grad_norm": grad_value,

            "tokens_per_second": speed,

            "gpu_memory_gb": mem,

            "fineweb_val_loss": None,

            "reasoning_val_loss": None,
        })

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    should_evaluate = (
        global_step % args.eval_every == 0
        or global_step == args.steps
    )

    if should_evaluate:

        last_fineweb_loss = evaluate_fineweb()

        write_log(
            f">>> step {global_step} | "
            f"FineWeb validation loss "
            f"{last_fineweb_loss:.4f}"
        )

        last_reasoning_loss = None

        if reasoning_data is not None:

            last_reasoning_loss = evaluate_reasoning()

            write_log(
                f">>> step {global_step} | "
                f"Reasoning validation loss "
                f"{last_reasoning_loss:.4f}"
            )

        # Update current metric record.

        if (
            training_history["metrics"]
            and training_history["metrics"][-1]["step"] == global_step
        ):

            training_history["metrics"][-1]["fineweb_val_loss"] = (
                last_fineweb_loss
            )

            training_history["metrics"][-1]["reasoning_val_loss"] = (
                last_reasoning_loss
            )

        # ----------------------------------------------------
        # BEST FINEWEB CHECKPOINT
        # ----------------------------------------------------

        if last_fineweb_loss < best_fineweb_loss:

            best_fineweb_loss = last_fineweb_loss

            save_checkpoint(
                best_fineweb_path,
                global_step
            )

            write_log(
                f"New best FineWeb validation loss: "
                f"{best_fineweb_loss:.4f}"
            )

        # ----------------------------------------------------
        # BEST REASONING CHECKPOINT
        # ----------------------------------------------------

        if (
            last_reasoning_loss is not None
            and last_reasoning_loss < best_reasoning_loss
        ):

            best_reasoning_loss = last_reasoning_loss

            save_checkpoint(
                best_reasoning_path,
                global_step
            )

            write_log(
                f"New best reasoning validation loss: "
                f"{best_reasoning_loss:.4f}"
            )

    # --------------------------------------------------------
    # LATEST CHECKPOINT
    # --------------------------------------------------------

    should_save = (
        global_step % args.save_every == 0
        or global_step == args.steps
    )

    if should_save:

        save_checkpoint(
            latest_path,
            global_step
        )

    # --------------------------------------------------------
    # SAVE METRICS
    # --------------------------------------------------------

    if should_evaluate or should_save:
        save_metrics()


# ============================================================
# 20. TRAINING COMPLETE
# ============================================================

training_history["end_time"] = datetime.now().isoformat()

save_metrics()

write_log("\nTraining completed successfully.")

write_log(f"Latest checkpoint: {latest_path}")

write_log(f"Best FineWeb checkpoint: {best_fineweb_path}")

write_log(f"Best reasoning checkpoint: {best_reasoning_path}")

write_log(f"Training log: {log_file}")

write_log(f"Training metrics: {log_json}")

write_log("\nFinal validation results:")

write_log(
    f"Best FineWeb validation loss: {best_fineweb_loss:.4f}"
)

write_log(
    f"Best reasoning validation loss: {best_reasoning_loss:.4f}"
)

## Part 1: Tokenizer + tokenization

### 1.1 Train the tokenizer from a streamed sample
Same settings as your `train_tokenizer.py` (byte-level BPE, 5 special tokens so `<eos>` stays id 2), now with a 32,768 vocab to match the model. Skipped if `tokenizer.json` already exists.

In [ ]:
#@title 1.1 Train tokenizer (streamed sample)
from datasets import load_dataset
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

SPECIALS = ["<pad>", "<bos>", "<eos>", "<think>", "</think>"]

if os.path.exists(TOK_PATH) and not FORCE_RETRAIN_TOKENIZER:
    print("Tokenizer already exists, skipping:", TOK_PATH)
else:
    ds_tok = load_dataset(DATASET, split="train", streaming=True)
    src_seen = Counter()

    def texts():
        n = 0
        for ex in ds_tok:
            t = ex["text"]
            if not isinstance(t, str) or not t.strip():
                continue
            if SOURCES and ex["source"] not in SOURCES:
                continue
            src_seen[ex["source"]] += 1
            yield t[:TOK_MAX_CHARS]
            n += 1
            if n >= TOK_TRAIN_DOCS:
                break

    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE, special_tokens=SPECIALS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True, min_frequency=2,
    )
    print("Training tokenizer on", f"{TOK_TRAIN_DOCS:,}", "streamed docs ...")
    t0 = time.time()
    tok.train_from_iterator(texts(), trainer=trainer)
    elapsed = time.time() - t0
    tok.save(TOK_PATH)
    with open(f"{PROJECT_DIR}/tokenizer/metadata.json", "w") as f:
        json.dump({"vocab_size": tok.get_vocab_size(),
                   "special_tokens": {t: tok.token_to_id(t) for t in SPECIALS},
                   "algorithm": "ByteLevel BPE", "trained_on": DATASET,
                   "docs": dict(src_seen), "training_time_minutes": round(elapsed/60, 2)}, f, indent=2)
    print(f"Done in {elapsed/60:.1f} min | vocab {tok.get_vocab_size()} | source mix {dict(src_seen)}")

# ---- checks (always run) ----
tok = Tokenizer.from_file(TOK_PATH)
print("Special token ids:", {t: tok.token_to_id(t) for t in SPECIALS})
assert tok.token_to_id("<eos>") == 2, "train_4k.py expects <eos> == 2"
assert tok.get_vocab_size() == VOCAB_SIZE, f"tokenizer vocab {tok.get_vocab_size()} != VOCAB_SIZE {VOCAB_SIZE}"
sample = "The cat sat on the mat. <think>2 + 2 = 4</think>"
enc = tok.encode(sample)
print("Tokens :", enc.tokens)
print("Round trip OK:", tok.decode(enc.ids, skip_special_tokens=False) == sample)

### 1.2 Stream, tokenize, write shards to Drive
- Streams the dataset; raw text never touches disk. Output is `train_shards/shard_00000.bin ...` (2B tokens each) plus `val.bin`.
- Every 100th document goes to validation (as in your `prep_data.py`), capped at `VAL_MAX_TOKENS`.
- The dataset cursor and token counts are saved every ~50M tokens. **If the session dies, re-run this cell**: it rolls the shard files back to the last saved point and continues.
- The progress line shows throughput and the recent source mix (if the mix drifts a lot over time, the stream is not shuffled and sequential training would see that drift).

In [ ]:
#@title 1.2 Tokenize to shards on Drive (resumable)
from itertools import islice
import numpy as np
from datasets import load_dataset
from tokenizers import Tokenizer

tok = Tokenizer.from_file(TOK_PATH)
EOS = tok.token_to_id("<eos>")
assert tok.get_vocab_size() <= 65536, "uint16 storage needs vocab <= 65536"
MAX_DOC_CHARS = 200_000   # guards against multi-MB documents
S = SHARD_TOKENS

def shard_path(k): return f"{SHARD_DIR}/shard_{k:05d}.bin"

def count_stored():
    k = total = 0
    while os.path.exists(shard_path(k)):
        n = os.path.getsize(shard_path(k)) // 2
        total += n
        if n < S: break
        k += 1
    return total

class ShardWriter:
    def __init__(self, total):
        self.k, self.off = divmod(total, S)
        j = self.k + 1
        while os.path.exists(shard_path(j)):      # drop anything beyond the resume point
            os.remove(shard_path(j)); j += 1
        p = shard_path(self.k)
        self.f = open(p, "r+b" if os.path.exists(p) else "wb")
        self.f.truncate(self.off * 2)
        self.f.seek(self.off * 2)
    def write(self, arr):
        i = 0
        while i < len(arr):
            n = min(S - self.off, len(arr) - i)
            arr[i:i+n].tofile(self.f); self.off += n; i += n
            if self.off == S: self._roll()
    def sync(self):
        self.f.flush(); os.fsync(self.f.fileno())
    def _roll(self):
        self.sync(); self.f.close()
        self.k += 1; self.off = 0
        self.f = open(shard_path(self.k), "wb")
    def close(self):
        self.f.close()

def write_manifest(done):
    m = {"shard_tokens": S, "planned_tokens": TRAIN_TOKENS_TARGET, "pattern": "shard_{:05d}.bin",
         "done": bool(done), "actual_tokens": state["train_tokens"]}
    tmp = f"{SHARD_DIR}/manifest.json.tmp"
    with open(tmp, "w") as f: json.dump(m, f)
    os.replace(tmp, f"{SHARD_DIR}/manifest.json")

# ---------- load / init state ----------
if os.path.exists(STATE_PATH):
    with open(STATE_PATH, "rb") as f:
        state = pickle.load(f)
    stored = count_stored()
    if stored < state["train_tokens"]:
        print(f"WARNING: shards hold {stored:,} tokens but state says {state['train_tokens']:,}. "
              f"Using the shards' length (a small gap in the stream is harmless).")
        state["train_tokens"] = stored
    have = (os.path.getsize(VAL_BIN) // 2) if os.path.exists(VAL_BIN) else 0
    if have > state["val_tokens"]:
        with open(VAL_BIN, "r+b") as f: f.truncate(state["val_tokens"] * 2)
    elif have < state["val_tokens"]:
        print("WARNING: val.bin shorter than state; using file length.")
        state["val_tokens"] = have
    print(f"Resuming: {state['docs']:,} docs, {state['train_tokens']:,} train tokens")
else:
    state = {"docs": 0, "train_tokens": 0, "val_tokens": 0, "ds_state": None, "src": {}, "done": False}
    open(VAL_BIN, "wb").close()

state["done"] = state["train_tokens"] >= TRAIN_TOKENS_TARGET

def save_state(ds, writer, f_val):
    writer.sync(); f_val.flush(); os.fsync(f_val.fileno())
    try:
        state["ds_state"] = ds.state_dict()
    except Exception:
        state["ds_state"] = None
    write_manifest(state["done"])
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "wb") as fh: pickle.dump(state, fh)
    os.replace(tmp, STATE_PATH)

if state["done"]:
    write_manifest(True)
    print("Already have enough train tokens.")
else:
    ds = load_dataset(DATASET, split="train", streaming=True)
    if state["docs"] > 0:
        resumed = False
        if state["ds_state"] is not None:
            try:
                ds.load_state_dict(state["ds_state"]); resumed = True
                print("Dataset cursor restored from saved state.")
            except Exception as e:
                print("Could not restore cursor (", e, ") - falling back to skip().")
        if not resumed:
            ds = ds.skip(state["docs"])
            print(f"Skipped {state['docs']:,} docs.")
    it = iter(ds)

    writer = ShardWriter(state["train_tokens"])
    f_val = open(VAL_BIN, "ab")
    write_manifest(False)
    t0 = last_print = time.time()
    start_tokens = state["train_tokens"]
    n_batches = 0
    window = Counter()

    try:
        while state["train_tokens"] < TRAIN_TOKENS_TARGET:
            rows = list(islice(it, BATCH_DOCS))
            if not rows:
                print("Dataset exhausted."); break
            items = []
            for ex in rows:
                doc_idx = state["docs"]; state["docs"] += 1
                text = ex["text"]
                if not isinstance(text, str) or not text.strip():
                    continue
                if SOURCES and ex["source"] not in SOURCES:
                    continue
                items.append((doc_idx, ex["source"], text[:MAX_DOC_CHARS]))

            encs = tok.encode_batch([t for _, _, t in items])
            tr, va, n_va = [], [], 0
            for (doc_idx, src, _), enc in zip(items, encs):
                arr = np.array(enc.ids + [EOS], dtype=np.uint16)
                if doc_idx % 100 == 0 and state["val_tokens"] + n_va < VAL_MAX_TOKENS:
                    va.append(arr); n_va += len(arr)
                else:
                    tr.append(arr)
                state["src"][src] = state["src"].get(src, 0) + 1
                window[src] += 1

            if tr:
                a = np.concatenate(tr); writer.write(a); state["train_tokens"] += len(a)
            if va:
                a = np.concatenate(va); a.tofile(f_val); state["val_tokens"] += len(a)
            n_batches += 1

            if n_batches % SAVE_STATE_EVERY_BATCHES == 0:
                save_state(ds, writer, f_val)

            now = time.time()
            if now - last_print > 60:
                rate = (state["train_tokens"] - start_tokens) / (now - t0)
                eta_h = (TRAIN_TOKENS_TARGET - state["train_tokens"]) / max(rate, 1) / 3600
                tot = sum(window.values()) or 1
                mix = " ".join(f"{k}:{100*v/tot:.0f}%" for k, v in window.most_common())
                print(f"{state['docs']:>12,} docs | {state['train_tokens']/1e9:7.3f}B / {TRAIN_TOKENS_TARGET/1e9:.0f}B tokens "
                      f"| {rate/1e6:.2f}M tok/s | eta {eta_h:.1f} h | mix {mix}", flush=True)
                last_print = now; window.clear()
        if state["train_tokens"] >= TRAIN_TOKENS_TARGET:
            state["done"] = True
        save_state(ds, writer, f_val)
    finally:
        writer.close(); f_val.close()

print(f"\ntrain tokens: {state['train_tokens']:,}")
print(f"val tokens  : {state['val_tokens']:,}")
print("source mix  :", state["src"])
print("done        :", state["done"])

## Part 2: Training (A100)

Starts as soon as the first shard is complete, trains with your `train_4k.py`, and writes checkpoints + logs to Drive.
Re-running the training cell after a disconnect resumes from `checkpoints/<TAG>_latest.pt`.

**About context length:** the model is configured for 16,384 positions, but `SEQ = 4096` here, the same as your earlier runs. Training at 16,384 from step 0 costs roughly twice the compute per token. Set `SEQ = 16384` (and `MICRO_BS = 1`, `ACCUM = 32` to keep 524,288 tokens/step) only if you really want that; changing it later requires a fresh run.

In [ ]:
#@title 2.1 Checks, run plan, local val copy, block order
import numpy as np

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python model_4k.py

# the first shard must be complete on Drive
p0 = f"{SHARD_DIR}/shard_00000.bin"
assert os.path.exists(f"{SHARD_DIR}/manifest.json") and os.path.exists(p0) and os.path.getsize(p0) == SHARD_TOKENS * 2, \
    "First shard is not finished yet - run/finish Part 1.2 first."
with open(STATE_PATH, "rb") as f:
    prep = pickle.load(f)
assert prep["val_tokens"] >= 0.98 * VAL_MAX_TOKENS, "Validation set not complete yet - let Part 1.2 run a bit longer."

# vocab must match the tokenizer
from tokenizers import Tokenizer
from model_4k import Config
assert Config().vocab_size == Tokenizer.from_file(TOK_PATH).get_vocab_size(), "Config.vocab_size != tokenizer vocab"

# run plan: fixed once, so every session uses identical settings
plan_now = {"steps": STEPS, "micro_bs": MICRO_BS, "accum": ACCUM, "seq": SEQ,
            "target_tokens": TRAIN_TOKENS_TARGET, "shard_tokens": SHARD_TOKENS}
if os.path.exists(PLAN_PATH):
    plan = json.load(open(PLAN_PATH))
    for k, v in plan_now.items():
        assert plan[k] == v, f"config changed since the run started: {k}: {plan[k]} -> {v}"
else:
    plan = dict(plan_now, val_tokens=prep["val_tokens"])
    json.dump(plan, open(PLAN_PATH, "w"), indent=2)
print("run plan:", plan)

# local copy of the (small) validation file, frozen to the size recorded in the plan
os.makedirs(LOCAL_DIR, exist_ok=True)
local_val = f"{LOCAL_DIR}/val.bin"
want = plan["val_tokens"] * 2
if not (os.path.exists(local_val) and os.path.getsize(local_val) == want):
    shutil.copyfile(VAL_BIN, local_val + ".tmp"); os.truncate(local_val + ".tmp", want); os.replace(local_val + ".tmp", local_val)

# block order: every 4096-token block of the 100B stream exactly once, in file order
BLOCKS_PATH = f"{PROJECT_DIR}/data/tokenized/blocks_seq{SEQ}.npy"
if not os.path.exists(BLOCKS_PATH):
    np.save(BLOCKS_PATH, np.arange((TRAIN_TOKENS_TARGET - 1) // SEQ, dtype=np.int32))
print("ready.")

In [ ]:
#@title 2.2 Train / resume
os.chdir(PROJECT_DIR)
latest = f"checkpoints/{TAG}_latest.pt"

cmd = ["python", "-u", "train_4k.py",
       "--steps", str(STEPS), "--micro_bs", str(MICRO_BS), "--accum", str(ACCUM), "--seq", str(SEQ),
       "--lr", str(LR), "--warmup_steps", str(WARMUP_STEPS),
       "--eval_every", str(EVAL_EVERY), "--save_every", str(SAVE_EVERY),
       "--unseen", BLOCKS_PATH, "--seed", str(SEED), "--tag", TAG,
       "--train_bin", SHARD_DIR, "--val_bin", local_val,
       "--reasoning_pct", str(REASONING_PCT)]
env = dict(os.environ, SEQUENTIAL_BLOCKS="1", SHARD_CACHE_DIR=SHARD_CACHE_DIR, PYTHONUNBUFFERED="1")

if os.path.exists(latest):
    cmd += ["--resume", latest]
    print("Resuming from", latest)
else:
    print("Starting a fresh run:", TAG)
print(" ".join(cmd), "\n")

proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        # full log is always in logs/<TAG>.log on Drive; here show only the useful lines
        if line.startswith("step"):
            try:
                if int(line.split()[1]) % 100 != 0:
                    continue
            except ValueError:
                pass
        print(line, end="")
    proc.wait()
except KeyboardInterrupt:
    proc.terminate(); proc.wait()
    print("\nStopped. Re-run this cell to resume from the latest checkpoint.")
print("exit code:", proc.returncode)

In [ ]:
#@title 2.3 Progress plot (can be run from another tab/session)
import matplotlib.pyplot as plt
h = json.load(open(f"{PROJECT_DIR}/logs/{TAG}.json"))
m = h["metrics"]
steps = [r["step"] for r in m]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(steps, [r["train_loss"] for r in m]); ax[0].set_title("train loss"); ax[0].set_xlabel("step")
v = [(r["step"], r["fineweb_val_loss"]) for r in m if r["fineweb_val_loss"] is not None]
if v:
    ax[1].plot(*zip(*v), marker="o"); ax[1].set_title("validation loss"); ax[1].set_xlabel("step")
plt.tight_layout(); plt.show()
print("last step:", steps[-1] if steps else None, "of", h["config"]["steps"])